In [10]:
import sys
sys.path.append('..')

from src.dataloader import DataLoader
from helpers.tfidf_vectorizer import TfidfVectorizer
from models.sklearn_decision_tree import SklearnDecisionTreeModel
from sklearn.metrics import accuracy_score

import numpy as np
import pandas as pd

# Configs
json_path = '../models/fitted_decision_tree.json'
train_data_path = '../src/data/train_data_raw.csv'
val_data_path = '../src/data/validation_data_raw.csv'
test_data_path = '../src/data/test_data_raw.csv'
random_seed = 42

hyperparams = {
    "random_state": random_seed,
    "max_depth": [3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 20],
    "min_samples_split": [2],
    "criterion": ["gini", "entropy"],
    "text_truncate_length": [3, 4, 5, 6, 7, 8],
    "normalize_vectors": [False]
}



In [11]:
def search_hyperparameters():
    best_acc = 0.0
    best_params = {}

    for max_depth in hyperparams["max_depth"]:
        for min_samples_split in hyperparams["min_samples_split"]:
            for criterion in hyperparams["criterion"]:
                for text_truncate_length in hyperparams["text_truncate_length"]:
                    for normalize_vectors in hyperparams["normalize_vectors"]:
                        dataloader = TfidfVectorizer(seed=random_seed, truncate_length=text_truncate_length)
                        dataloader.build_vocab(train_data_path, verbose=False)
                        X_train, y_train = dataloader.generate_Xt(train_data_path, normalize=normalize_vectors, verbose=False)
                        X_val, y_val = dataloader.generate_Xt(val_data_path, normalize=normalize_vectors, verbose=False)

                        model = SklearnDecisionTreeModel(
                            max_depth=max_depth,
                            min_samples_split=min_samples_split,
                            criterion=criterion,
                            random_state=random_seed
                        )
                        model.fit(X_train, y_train)
                        y_val_pred = model.predict(X_val)
                        val_accuracy = accuracy_score(y_val, y_val_pred)

                        if val_accuracy > best_acc:
                            best_acc = val_accuracy
                            best_params = {
                                "max_depth": max_depth,
                                "min_samples_split": min_samples_split,
                                "criterion": criterion,
                                "text_truncate_length": text_truncate_length,
                                "normalize_vectors": normalize_vectors
                            }

                        print(f"Params: max_depth={max_depth}, min_samples_split={min_samples_split}, "
                              f"criterion={criterion}, text_truncate_length={text_truncate_length}, "
                              f"normalize_vectors={normalize_vectors} \n\t => Validation Accuracy: {val_accuracy:.4f}")

    return best_params

In [23]:
dataloader = TfidfVectorizer(seed=random_seed, truncate_length=3)
dataloader.build_vocab(train_data_path, verbose=False)
X_train, y_train = dataloader.generate_Xt(train_data_path, normalize=False, verbose=False)
X_val, y_val = dataloader.generate_Xt(val_data_path, normalize=False, verbose=False)

print(dataloader.get_vocab_size())

8050


In [24]:
model = SklearnDecisionTreeModel(max_depth=4,
                                 min_samples_split=2,
                                 random_state=hyperparams["random_state"],
                                 criterion='entropy')

In [25]:
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Tree depth: {model.get_depth()}")
print(f"Number of leaves: {model.get_n_leaves()}")
print(f"Number of nodes: {model.node_count()}")

Accuracy: 0.6133
Tree depth: 4
Number of leaves: 15
Number of nodes: 29


In [16]:
best_hyperparams = search_hyperparameters()
print("Best Hyperparameters:")
for param, value in best_hyperparams.items():
    print(f"{param}: {value}")

Params: max_depth=3, min_samples_split=2, criterion=gini, text_truncate_length=3, normalize_vectors=False 
	 => Validation Accuracy: 0.6000
Params: max_depth=3, min_samples_split=2, criterion=gini, text_truncate_length=4, normalize_vectors=False 
	 => Validation Accuracy: 0.6000
Params: max_depth=3, min_samples_split=2, criterion=gini, text_truncate_length=5, normalize_vectors=False 
	 => Validation Accuracy: 0.5933
Params: max_depth=3, min_samples_split=2, criterion=gini, text_truncate_length=6, normalize_vectors=False 
	 => Validation Accuracy: 0.6000
Params: max_depth=3, min_samples_split=2, criterion=gini, text_truncate_length=7, normalize_vectors=False 
	 => Validation Accuracy: 0.6000
Params: max_depth=3, min_samples_split=2, criterion=gini, text_truncate_length=8, normalize_vectors=False 
	 => Validation Accuracy: 0.6000
Params: max_depth=3, min_samples_split=2, criterion=entropy, text_truncate_length=3, normalize_vectors=False 
	 => Validation Accuracy: 0.6067
Params: max_depth